# Jina: обучение на всём train и предсказания



## 1. Установка библиотек


In [ ]:
%pip install -q "transformers==5.22.1" "accelerate==1.15.0" "einops==0.8.1"
%pip install -q --no-deps "torchao==0.18.0"


## 2. Импорты, Диск и общие параметры

Исходная модель и её ревизия, обработка текста, число эпох и learning rate одинаковые. Настройки batch и папки результатов находятся внутри каждого раздела обучения.


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import gc, html, json, math, re, zipfile
from pathlib import Path
import tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
from tqdm.auto import tqdm
from transformers import (
    AutoConfig, AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed,
)
import transformers.models.xlm_roberta.modeling_xlm_roberta as xlmr

assert torch.cuda.is_available(), "Включи GPU в Colab."

ROOT = Path(tempfile.gettempdir()) / "pmmo_ranknet"
CACHE = str(ROOT / "hf_cache")
NAME = "jinaai/jina-reranker-v2-base-multilingual"
REVISION = "9cfeff2df7d40d1b78e75e5e9cebec92a99813c9"

EPOCHS, MAX_LENGTH, LR, SEED = 2, 512, 5e-6, 42
PRED_BATCH = 8
BF16 = torch.cuda.is_bf16_supported()
AMP = torch.bfloat16 if BF16 else torch.float16

if not hasattr(xlmr, "create_position_ids_from_input_ids"):
    xlmr.create_position_ids_from_input_ids = (
        xlmr.XLMRobertaEmbeddings.create_position_ids_from_input_ids
    )
set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0))
print("Проект:", ROOT)


Mounted at /content/drive
GPU: Tesla T4
Проект: /content/drive/MyDrive/PMMO_ranking_v2


## 3. Поиск архива, однократное чтение данных и очистка

Архив ищется в /content, текущей и родительской папках,
в папке data внутри ROOT и в путях Kaggle.
CSV читаются непосредственно из ZIP.

Используется весь train. Выполняются нормализация пробелов,
удаление невидимых символов и очистка HTML в text.
Заголовок и содержание объединяются в document.
Регистр, числа и отрицания сохраняются.

In [ ]:
ZIP_NAME = "amml-ranking-competition-2026.zip"
possible_zip_paths = [
    Path("/content") / ZIP_NAME,
    Path.cwd() / ZIP_NAME,
    Path.cwd().parent / ZIP_NAME,
    ROOT / "data" / ZIP_NAME,
    Path("/kaggle/working") / ZIP_NAME,
    Path("/kaggle/input") / ZIP_NAME,
]
if Path("/kaggle/input").exists():
    possible_zip_paths.extend(Path("/kaggle/input").rglob(ZIP_NAME))

zip_path = next((p for p in possible_zip_paths if p.is_file()), None)
if zip_path is None:
    raise FileNotFoundError(
        f"Архив {ZIP_NAME} не найден. Загрузи его в /content/, "
        "положи рядом с ноутбуком или в папку data проекта на Диске."
    )
print("Архив найден:", zip_path)

types = {c: str for c in ["id", "query_id", "document_id", "article_id"]}
tables = {}
with zipfile.ZipFile(zip_path) as archive:
    for filename in ["train.csv", "test.csv", "sample_submission.csv"]:
        members = [name for name in archive.namelist() if Path(name).name == filename]
        assert len(members) == 1, f"Не найден однозначный {filename}"
        with archive.open(members[0]) as source:
            tables[filename] = pd.read_csv(source, dtype=types)
train = tables["train.csv"]
test = tables["test.csv"]
sample = tables["sample_submission.csv"]

tag_pattern = re.compile(
    r"</?(?:p|br|div|span|b|i|a|strong|em|sup|sub|ul|ol|li|h[1-6]|"
    r"table|tbody|tr|td|th|blockquote)(?:\s[^<>]*?)?\s*/?>", re.IGNORECASE,
)

def remove_html(text):
    text = html.unescape(html.unescape(text))
    return re.sub(r"\s+", " ", tag_pattern.sub(" ", text)).strip()

for frame in [train, test]:
    assert frame["id"].notna().all() and frame["id"].is_unique
    assert frame[["query_id", "query", "text"]].notna().all().all()
    for column in ["query", "title", "text"]:
        frame[column] = (
            frame[column].fillna("")
            .str.replace("[\u200b\ufeff]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.strip()
        )
    frame["text"] = frame["text"].map(remove_html)
    frame["document"] = (frame["title"] + "\n" + frame["text"]).str.strip()
    assert frame["query"].str.len().gt(0).all()
    assert frame.groupby("query_id")["query"].nunique().max() == 1

assert len(train) == 31530 and set(train["relevance"].unique()) == {0, 1}
assert set(train["query_id"]).isdisjoint(test["query_id"])
assert sample["id"].is_unique and set(sample["id"]) == set(test["id"])
assert sample.columns.tolist() == ["id", "prediction"]
print("Train:", len(train), "| запросов:", train["query_id"].nunique(), "| test:", len(test))


Архив найден: /content/drive/MyDrive/PMMO_ranking_v2/data/amml-ranking-competition-2026.zip
Train: 31530 | запросов: 3042 | test: 13507


## 4. Jina + RankNet

Исходная предобученная Jina дообучается с функцией потерь RankNet.

Веса сохраняются после каждой эпохи в
ROOT/runs/jina_all_ranknet_html_2ep_v1/checkpoint-*.
Итоговые предсказания — в submission_ranknet.csv той же папки.
ROOT находится во временной папке вычислительной среды.


### 5.1. Папка результатов и batch


In [ ]:
RUN_DIR = ROOT / "runs/jina_all_ranknet_html_2ep_v1"
SUBMISSION = RUN_DIR / "submission_ranknet.csv"
BATCH, ACCUM = 1, 4
DONE = SUBMISSION.exists()
RUN_DIR.mkdir(parents=True, exist_ok=True)
assert train.groupby("query_id")["relevance"].nunique().eq(2).all()
print("Результаты:", RUN_DIR)


Результаты: /content/drive/MyDrive/PMMO_ranking_v2/runs/jina_all_ranknet_html_2ep_v1


### 5.2. Лосс RankNet

Один элемент обучения — запрос со всеми его документами. Сравниваем все пары «релевантный — нерелевантный» внутри запроса. За один проход модели обрабатываются два документа, лосс считается по всей группе. Градиенты накапливаются по четырём запросам.


In [ ]:
def rank_collator(items):
    assert len(items) == 1
    return {"query": items[0]}

class RankNetTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        query = inputs["query"]
        parts = []
        for start in range(0, len(query["rows"]), 2):
            tokens = tokenizer.pad(
                query["rows"][start:start + 2],
                padding=True, pad_to_multiple_of=8, return_tensors="pt",
            ).to(self.args.device)
            parts.append(model(**tokens).logits.reshape(-1))
        scores = torch.cat(parts).float()
        positive = torch.as_tensor(query["target"], device=scores.device) == 1
        loss = F.softplus(
            scores[~positive][None, :] - scores[positive][:, None]
        ).mean()
        return (loss, {"logits": scores[:, None]}) if return_outputs else loss


### 5.3. Исходная Jina → две эпохи RankNet

Используются все 3042 запроса и все 31 530 строк. Число обновлений отличается от обучения по отдельным парам, поскольку batch здесь измеряется в запросах.


In [ ]:
model, trainer = None, None
if not DONE:
    checkpoints = list(RUN_DIR.glob("checkpoint-*"))
    finished = [
        path for path in checkpoints
        if (path / "model.safetensors").exists()
        and (path / "trainer_state.json").exists()
        and float(json.loads((path / "trainer_state.json").read_text()).get("epoch") or 0) >= EPOCHS
    ]
    assert finished or not checkpoints, (
        f"В {RUN_DIR} есть незавершённое обучение без состояния оптимизатора. "
        "Для нового полного запуска измени RUN_DIR. Сохранённые веса остаются."
    )
    source = max(finished, key=lambda p: int(p.name.split("-")[-1])) if finished else NAME
    options = {"revision": REVISION} if str(source) == NAME else {}
    tokenizer = AutoTokenizer.from_pretrained(
        source, cache_dir=CACHE, trust_remote_code=True, **options,
    )
    config = AutoConfig.from_pretrained(
        source, cache_dir=CACHE, trust_remote_code=True, **options,
    )
    config.use_flash_attn = False
    config.use_cache = False
    assert config.num_labels == 1
    gc.collect()
    torch.cuda.empty_cache()
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        source, config=config, cache_dir=CACHE, trust_remote_code=True,
        torch_dtype=torch.float32, attn_implementation="eager", **options,
    )
    model.requires_grad_(True)
    model.roberta.encoder.gradient_checkpointing = True

    if not finished:
        encoded = tokenizer(
            train["query"].tolist(), train["document"].tolist(),
            truncation="only_second", max_length=MAX_LENGTH, padding=False,
        )
        rows = [
            {key: values[i] for key, values in encoded.items()}
            for i in range(len(train))
        ]

        train_data = []
        for _, group in train.groupby("query_id", sort=True):
            group = group.sort_values("id")
            train_data.append({
                "rows": [rows[i] for i in group.index],
                "target": group["relevance"].tolist(),
            })
        assert sum(len(query["rows"]) for query in train_data) == len(train)

        steps_per_epoch = math.ceil(len(train_data) / (BATCH * ACCUM))
        args = TrainingArguments(
            output_dir=str(RUN_DIR), num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
            learning_rate=LR, warmup_steps=math.ceil(steps_per_epoch * EPOCHS * 0.1),
            weight_decay=0.01, optim="adamw_torch", lr_scheduler_type="linear",
            fp16=not BF16, bf16=BF16,
            eval_strategy="no", save_strategy="epoch",
            save_only_model=True, save_total_limit=2,
            logging_steps=50, report_to="none", seed=SEED, data_seed=SEED,
            remove_unused_columns=False, dataloader_num_workers=0,
        )
        trainer = RankNetTrainer(
            model=model, args=args, train_dataset=train_data,
            processing_class=tokenizer, data_collator=rank_collator,
        )
        trainer.model_accepts_loss_kwargs = False
        trainer.train()
        model = trainer.model
    else:
        print("Две эпохи уже завершены. Предсказываем из:", source)
else:
    print("CSV уже готов. Обучение пропускаем:", SUBMISSION)


Loading weights:   0%|          | 0/153 [00:00<?, ?it/s]

Step,Training Loss
50,0.223242
100,0.209059
150,0.165191
200,0.180570
250,0.176555
300,0.170010
350,0.124863
400,0.126416
450,0.147536
500,0.161801


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### 5.4. Предсказания → второй CSV

Ответы идут в порядке `sample_submission.csv`. Скачай готовый CSV из Google Диска после возвращения.


In [ ]:
if not DONE:
    model.to("cuda").eval()
    scores = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(test), PRED_BATCH), desc="Предсказания RankNet"):
            part = test.iloc[start:start + PRED_BATCH]
            inputs = tokenizer(
                part["query"].tolist(), part["document"].tolist(),
                truncation="only_second", max_length=MAX_LENGTH,
                padding=True, pad_to_multiple_of=8, return_tensors="pt",
            ).to("cuda")
            with torch.autocast("cuda", dtype=AMP):
                values = model(**inputs).logits.reshape(-1)
            scores.extend(values.float().cpu().tolist())
    assert len(scores) == len(test) and np.isfinite(scores).all()
    predictions = test[["id"]].assign(prediction=scores)
    submission = sample[["id"]].merge(
        predictions, on="id", how="left", sort=False, validate="one_to_one",
    )
    assert submission["id"].tolist() == sample["id"].tolist()
    assert np.isfinite(submission["prediction"].to_numpy()).all()
    temporary = SUBMISSION.with_suffix(".tmp")
    submission.to_csv(temporary, index=False)
    temporary.replace(SUBMISSION)
else:
    submission = pd.read_csv(SUBMISSION, dtype={"id": str})

assert submission.columns.tolist() == sample.columns.tolist()
assert submission["id"].tolist() == sample["id"].tolist()
assert np.isfinite(submission["prediction"].to_numpy()).all()
print("Готово:", SUBMISSION, "| строк:", len(submission))
display(submission.head())

del trainer, model
gc.collect()
torch.cuda.empty_cache()
